# Lesson 10 Lab — SM Resources: Occupancy, Registers, and Banks

**Puzzle:** Does maximizing occupancy guarantee a fast kernel, and can shared-memory bank conflicts matter even when data stays on chip?

This notebook retains one complete RTX 5090 execution.


## Why this matters

An SM admits thread blocks only while registers, shared memory, warp slots, and block slots are available. Occupancy measures resident warps relative to a hardware maximum; it helps hide latency but does not guarantee useful instructions, coalescing, or balanced pipelines. Shared memory is partitioned into banks, so simultaneous warp addresses that map to the same bank may be serialized unless the access is a supported broadcast.


## 0. Predict before running

1. Predict which resource limits each candidate block.
2. Predict the maximum bank multiplicity for strides 1, 2, and 32.
3. Explain why 100% modeled occupancy is not a performance promise.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

The lab reads CUDA device properties, evaluates an explicit resource-budget formula for several candidate kernels, and maps warp lanes to 32 illustrative banks for different strides. These are capacity and address models. They do not expose per-kernel register allocation or prove a native bank conflict; those require compiled kernel metadata and profiler counters.

- Occupancy is constrained by the tightest resident resource.
- Higher occupancy can trade against registers or shared-memory reuse.
- Bank conflict is an address-mapping property inside a warp.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["block request"] --> B["thread/warp slots"]
  A --> C["register budget"]
  A --> D["shared-memory budget"]
  B --> E["resident blocks"]
  C --> E
  D --> E
  E --> F["scheduler hides latency"]
```


## 3. Inspect the visual boundary

![Conceptual SM compute partition](../assets/SM_compute_partition_circuit_structure.png)

- [Printable NoC and SM diagrams](../assets/NoC_and_SM_circuit_structures_A4_portrait.pdf)

These are conceptual teaching diagrams. They explain the named data path and are not die-accurate schematics of a particular commercial GPU.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 10
LESSON_TITLE = 'SM Resources: Occupancy, Registers, and Banks'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260823
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | moderate threads, registers, and shared memory per block |
| Candidate | register-heavy, shared-heavy, and conflicting-stride cases |
| Held constant | declared resource limits, 32-lane warp, and 32 illustrative banks |
| Measurements | resident blocks/warps, occupancy bound, and bank multiplicity |
| Evidence | `capacity-model` |

**Experiment:** Compute resource-limited occupancy and shared-memory bank mappings.


## 6. Inspect the code

The calculation takes the minimum block limit from threads, registers, shared memory, and block slots. A separate mapping counts bank IDs for each stride. Both tables remain inspectable and architecture assumptions are printed.

Do not run until the code matches the frozen table.


In [2]:
limits = {
    "max_threads_sm": 2048, "max_warps_sm": 64, "max_blocks_sm": 16,
    "registers_sm": 65536, "shared_bytes_sm": 64 * 1024,
}
cases = {
    "balanced": {"threads": 256, "registers_thread": 32, "shared_bytes": 8 * 1024},
    "register_heavy": {"threads": 256, "registers_thread": 112, "shared_bytes": 8 * 1024},
    "shared_heavy": {"threads": 256, "registers_thread": 32, "shared_bytes": 40 * 1024},
}

def occupancy(case):
    warps = math.ceil(case["threads"] / 32)
    block_limits = {
        "threads": limits["max_threads_sm"] // case["threads"],
        "warps": limits["max_warps_sm"] // warps,
        "registers": limits["registers_sm"] // (case["threads"] * case["registers_thread"]),
        "shared": limits["shared_bytes_sm"] // case["shared_bytes"],
        "blocks": limits["max_blocks_sm"],
    }
    resident_blocks = min(block_limits.values())
    return {"resident_blocks": resident_blocks, "resident_warps": resident_blocks * warps,
            "occupancy": resident_blocks * warps / limits["max_warps_sm"],
            "limiting_resources": [k for k, v in block_limits.items() if v == resident_blocks],
            "block_limits": block_limits}

occ = {name: occupancy(case) for name, case in cases.items()}
bank_multiplicity = {}
for stride in (1, 2, 4, 8, 16, 32):
    counts = Counter((lane * stride) % 32 for lane in range(32))
    bank_multiplicity[f"stride_{stride}"] = max(counts.values())
metrics = {
    "model_limits": limits, "device_name": ENV["gpu"],
    "occupancy": {name: value["occupancy"] for name, value in occ.items()},
    "occupancy_details": occ, "bank_multiplicity": bank_multiplicity,
}
analysis = (
    f"Modeled occupancy was {occ['balanced']['occupancy']:.1%}, "
    f"{occ['register_heavy']['occupancy']:.1%}, and {occ['shared_heavy']['occupancy']:.1%}; "
    f"stride 32 mapped all lanes to one illustrative bank (multiplicity "
    f"{bank_multiplicity['stride_32']})."
)
print(json.dumps(metrics, indent=2))


{
  "model_limits": {
    "max_threads_sm": 2048,
    "max_warps_sm": 64,
    "max_blocks_sm": 16,
    "registers_sm": 65536,
    "shared_bytes_sm": 65536
  },
  "device_name": "NVIDIA GeForce RTX 5090",
  "occupancy": {
    "balanced": 1.0,
    "register_heavy": 0.25,
    "shared_heavy": 0.125
  },
  "occupancy_details": {
    "balanced": {
      "resident_blocks": 8,
      "resident_warps": 64,
      "occupancy": 1.0,
      "limiting_resources": [
        "threads",
        "warps",
        "registers",
        "shared"
      ],
      "block_limits": {
        "threads": 8,
        "warps": 8,
        "registers": 8,
        "shared": 8,
        "blocks": 16
      }
    },
    "register_heavy": {
      "resident_blocks": 2,
      "resident_warps": 16,
      "occupancy": 0.25,
      "limiting_resources": [
        "registers"
      ],
      "block_limits": {
        "threads": 8,
        "warps": 8,
        "registers": 2,
        "shared": 8,
        "blocks": 16
      }
    },
    "

## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Balanced occupancy bound | 100.00% |
| Register-heavy occupancy | 25.00% |
| Shared-heavy occupancy | 12.50% |
| Stride-1 bank multiplicity | 1 |
| Stride-32 bank multiplicity | 32 |


## 8. Explain rather than overclaim

Modeled occupancy was 100.0%, 25.0%, and 12.5%; stride 32 mapped all lanes to one illustrative bank (multiplicity 32).

**Evidence boundary:** Measured environment facts feed explicit capacity or Roofline arithmetic. Declared hierarchy and resource fields remain assumptions until native counters confirm them.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 10, "title": 'SM Resources: Occupancy, Registers, and Banks', "environment": ENV,
    "evidence_label": 'capacity-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Use occupancy and bank models to choose experiments, then accept an optimization only after native kernel timing and counters confirm the suspected limit.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 10,
  "title": "SM Resources: Occupancy, Registers, and Banks",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260823
  },
  "evidence_label": "capacity-model",
  "metrics": {
    "model_limits": {
      "max_threads_sm": 2048,
      "max_warps_sm": 64,
      "max_blocks_sm": 16,
      "registers_sm": 65536,
      "shared_bytes_sm": 65536
    },
    "device_name": "NVIDIA GeForce RTX 5090",
    "occupancy": {
      "balanced": 1.0,
      "register_heavy": 0.25,
      "shared_heavy": 0.125
    },
    "occupancy_details": {
      "balanced": {
        "resident_blocks": 8,
        "resident_warps": 64,
        "occupancy": 1.0,
        "limiting_resources": [
          "threads",
          "warps",
          "registers",
          "shared"
        ],
        "block_limits": {
          "threads": 8,
          "warps": 8,
          "regis

## 10. Make the decision

> Use occupancy and bank models to choose experiments, then accept an optimization only after native kernel timing and counters confirm the suspected limit.

**Failure analysis:** The model uses declared illustrative resource limits because PyTorch does not expose every SM scheduling field uniformly. Broadcast rules and bank width can change the naive multiplicity interpretation.


## 11. Extend the evidence

Compile two CUDA kernels with `-Xptxas -v`, record registers/shared memory, and profile achieved occupancy plus bank-conflict counters.

See [`README.md`](README.md) for the full explanation and references.
